# IDX Stock Prediction 2026: full pipeline on Colab

Predicts the next-day return of **ASII**, **BBRI** and **TLKM** for every trading day in 2026, and checks whether
the predictions beat simple baselines and buy & hold.

No GPU is needed. The whole notebook runs in about 2 minutes on a normal Colab CPU runtime.
Results (metrics, figures, predictions) land in `results/`. Commit that folder back to the repo afterwards.

In [ ]:
!git clone https://github.com/haikalfr04/Stocks-Prediction.git
%cd Stocks-Prediction
!pip install -q -r requirements.txt

## 1. Look at the data directly from Yahoo Finance

The `yfinance` package downloads stock data from Yahoo Finance straight into a pandas DataFrame. Stocks listed on
the IDX use the `.JK` suffix. These cells only display the data. Section 2 saves it to Excel for the experiment.

In [ ]:
import pandas as pd
import yfinance as yf

tickers = {"BBRI": "BBRI.JK", "ASII": "ASII.JK", "TLKM": "TLKM.JK"}

data = {}
for code, ticker in tickers.items():
    # auto_adjust=False keeps both Close (the price shown on the IDX) and Adj Close
    # (adjusted for dividends and stock splits).
    df = yf.download(ticker, start="2015-01-01", auto_adjust=False, progress=False)
    if isinstance(df.columns, pd.MultiIndex):  # newer yfinance versions add a ticker level
        df.columns = df.columns.get_level_values(0)
    data[code] = df
    print(f"{code}: {len(df):,} trading days, {df.index[0]:%Y-%m-%d} to {df.index[-1]:%Y-%m-%d}")

The last 5 trading days of each stock (Open, High, Low, Close, Adj Close, Volume):

In [ ]:
for code, df in data.items():
    print(code)
    display(df.tail())

Closing prices of the three stocks side by side, and the summary statistics for 2026:

In [ ]:
close = pd.DataFrame({code: df["Close"] for code, df in data.items()})
display(close.tail(10))
display(close.loc["2026"].describe().round(0))

In [ ]:
ax = close.plot(subplots=True, figsize=(10, 7), title="Closing price (Rp)", legend=True)

Return in 2026 so far, and the daily return statistics:

In [ ]:
last_2025 = close.loc[:"2025"].iloc[-1]
latest = close.iloc[-1]
daily_return = close.pct_change()

summary = pd.DataFrame({
    "Close end of 2025": last_2025,
    "Latest close": latest,
    "Return 2026": (latest / last_2025 - 1).map("{:+.1%}".format),
    "Average daily return": daily_return.mean().map("{:+.3%}".format),
    "Daily volatility": daily_return.std().map("{:.2%}".format),
    "Days with no change": (daily_return == 0).mean().map("{:.1%}".format),
})
display(summary)

Company information and recent dividends. Some fields can be empty when Yahoo Finance has no value for them.

In [ ]:
rows = []
for code, ticker in tickers.items():
    info = yf.Ticker(ticker).info
    rows.append({
        "Code": code,
        "Name": info.get("longName"),
        "Sector": info.get("sector"),
        "Industry": info.get("industry"),
        "Market cap (Rp)": info.get("marketCap"),
        "P/E": info.get("trailingPE"),
        "52-week low": info.get("fiftyTwoWeekLow"),
        "52-week high": info.get("fiftyTwoWeekHigh"),
    })
display(pd.DataFrame(rows).set_index("Code"))

In [ ]:
for code, ticker in tickers.items():
    print(code, "dividends per share (Rp):")
    display(yf.Ticker(ticker).dividends.tail(5))

## 2. Download the data

Downloads daily prices since 2015 from Yahoo Finance into `data/idx_prices.xlsx`, with one sheet per stock
(Date, Open, High, Low, Close, Adj Close, Volume) and an `Info` sheet.

In [ ]:
!python -m src.download

In [ ]:
import pandas as pd
prices = pd.read_excel("data/idx_prices.xlsx", sheet_name=None, index_col=0)
display(prices["Info"])
display(prices["BBRI"].tail())

Optional: download the Excel file to open it on your computer.

In [ ]:
from google.colab import files
files.download("data/idx_prices.xlsx")

## 3. Exploratory data analysis

Summary statistics, price history, the distribution of daily returns, and their autocorrelation. Autocorrelation
close to zero means past returns say little about the next return, which already hints at how hard the task is.

In [ ]:
!python -m src.eda
from IPython.display import Image, Markdown, display
display(Markdown(open("results/eda_stats.md").read()))
for name in ["price_history", "return_distribution", "autocorrelation"]:
    display(Image(f"results/figures/{name}.png", width=800))

## 4. Walk-forward prediction for 2026

Models are trained only on data before the day they predict. The first model is trained on 2015-2025 data, predicts
the next 21 trading days, and is then retrained with those days added. This repeats until the end of the data.

Five models are compared: three baselines (random walk, historical mean, momentum), Ridge regression and LightGBM.
Each prediction is also used in a long-or-cash trading strategy with IDX fees, compared with buy & hold.

In [ ]:
!python -m src.run

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("results/comparison.md").read()))

In [ ]:
for name in ["model_comparison", "ASII_2026_predictions", "BBRI_2026_predictions", "TLKM_2026_predictions",
             "equity_curves", "monthly_accuracy", "feature_importance"]:
    display(Image(f"results/figures/{name}.png", width=800))

Daily predictions for 2026 (one row per trading day) are saved in `results/predictions/`.

In [ ]:
daily = pd.read_csv("results/predictions/BBRI_2026.csv", index_col=0, parse_dates=True)
daily[["prev_close", "actual_close", "pred_close_lightgbm", "actual_return", "pred_return_lightgbm"]].tail(10)

## 5. Predict the next trading day

LightGBM trained on all available history predicts the day after the last date in the data.
For learning purposes only. This is not investment advice.

In [ ]:
!python -m src.predict

## 6. Download the results to commit them to GitHub

In [ ]:
!zip -qr results.zip results data/idx_prices.xlsx
from google.colab import files
files.download("results.zip")